# Compile ONNX to TVM

Import the ONNX model into Relay, compile for CPU (`llvm`, opt level 3), then compare TVM against ONNX Runtime on accuracy, latency, and graph structure.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import random
import time

import numpy as np
import onnx
import onnxruntime as ort
import tvm
from tvm import relay
from tvm.contrib import graph_executor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
SEED = 42
BATCH_SIZE_EVAL = 1
TARGET = "llvm"
OPT_LEVEL = 3
TIMING_NUM_SAMPLES = 200
TIMING_NUM_WARMUP_RUNS = 10

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "artifacts" / "data"
ONNX_PATH = PROJECT_ROOT / "artifacts" / "onnx" / "lenet_mnist.onnx"
TVM_DIR = PROJECT_ROOT / "artifacts" / "tvm"
TVM_LIB_PATH = TVM_DIR / "lenet_mnist.tar"
TVM_GRAPH_PATH = TVM_DIR / "lenet_mnist_graph.json"
TVM_PARAMS_PATH = TVM_DIR / "lenet_mnist_params.bin"
TVM_RELAY_PATH = TVM_DIR / "lenet_mnist_relay.txt"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results" / "onnx_vs_tvm_verification.json"
compile_time_s = None

for path in [TVM_DIR, RESULTS_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

if not ONNX_PATH.exists():
    raise FileNotFoundError(f"Missing ONNX model: {ONNX_PATH}")

random.seed(SEED)
np.random.seed(SEED)

print(f"ONNX model: {ONNX_PATH}")
print(f"TVM target: {TARGET}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

test_dataset = datasets.MNIST(root=str(DATA_ROOT), train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)

print(f"Test samples: {len(test_dataset)}")
print(f"Evaluation batch size: {BATCH_SIZE_EVAL}")

In [ ]:
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

shape_dict = {"input": (1, 1, 28, 28)}
relay_mod, relay_params = relay.frontend.from_onnx(
    onnx_model,
    shape=shape_dict,
    freeze_params=True,
)

TVM_RELAY_PATH.write_text(relay_mod.astext(show_meta_data=False), encoding="utf-8")

compile_start = time.perf_counter()

with tvm.transform.PassContext(opt_level=OPT_LEVEL):
    compiled = relay.build(relay_mod, target=TARGET, params=relay_params)

compile_time_s = time.perf_counter() - compile_start

compiled.export_library(str(TVM_LIB_PATH))
TVM_GRAPH_PATH.write_text(compiled.get_graph_json(), encoding="utf-8")
TVM_PARAMS_PATH.write_bytes(relay.save_param_dict(compiled.get_params()))

print(f"Saved TVM library to: {TVM_LIB_PATH}")
print(f"Saved TVM graph to: {TVM_GRAPH_PATH}")
print(f"Saved TVM params to: {TVM_PARAMS_PATH}")
print(f"Saved Relay text to: {TVM_RELAY_PATH}")
print(f"TVM compile time (s): {compile_time_s:.3f}")

In [ ]:
ort_session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
ort_input_name = ort_session.get_inputs()[0].name

tvm_device = tvm.cpu(0)
tvm_lib = tvm.runtime.load_module(str(TVM_LIB_PATH))
tvm_module = graph_executor.create(TVM_GRAPH_PATH.read_text(encoding="utf-8"), tvm_lib, tvm_device)
tvm_module.load_params(TVM_PARAMS_PATH.read_bytes())

print(f"ONNX input name: {ort_input_name}")

In [ ]:
def run_onnx(inputs: np.ndarray) -> np.ndarray:
    return ort_session.run(None, {ort_input_name: inputs.astype(np.float32)})[0]

def run_tvm(inputs: np.ndarray) -> np.ndarray:
    tvm_module.set_input("input", tvm.nd.array(inputs.astype(np.float32), device=tvm_device))
    tvm_module.run()
    return tvm_module.get_output(0).numpy()

def count_onnx_parameters(model: onnx.ModelProto) -> int:
    return int(sum(int(np.prod(initializer.dims)) for initializer in model.graph.initializer))

def get_file_size_bytes(path: Path) -> int:
    return int(path.stat().st_size)

def count_relay_calls(expr) -> int:
    class CallCounter(tvm.relay.ExprVisitor):
        def __init__(self):
            super().__init__()
            self.count = 0

        def visit_call(self, call):
            self.count += 1
            super().visit_call(call)

    counter = CallCounter()
    counter.visit(expr)
    return int(counter.count)

def build_model_summary() -> dict:
    op_counts = Counter(node.op_type for node in onnx_model.graph.node)
    graph_json = json.loads(TVM_GRAPH_PATH.read_text(encoding="utf-8"))
    relay_text = TVM_RELAY_PATH.read_text(encoding="utf-8")
    tvm_artifact_sizes = {
        "library_bytes": get_file_size_bytes(TVM_LIB_PATH),
        "graph_json_bytes": get_file_size_bytes(TVM_GRAPH_PATH),
        "params_bytes": get_file_size_bytes(TVM_PARAMS_PATH),
        "relay_text_bytes": get_file_size_bytes(TVM_RELAY_PATH),
    }
    tvm_artifact_sizes["total_bytes"] = int(sum(tvm_artifact_sizes.values()))

    return {
        "onnx_file_bytes": get_file_size_bytes(ONNX_PATH),
        "onnx_num_graph_nodes": int(len(onnx_model.graph.node)),
        "onnx_num_initializers": int(len(onnx_model.graph.initializer)),
        "onnx_num_parameters": count_onnx_parameters(onnx_model),
        "onnx_op_type_counts": dict(sorted(op_counts.items())),
        "tvm_compile_time_s": float(compile_time_s),
        "relay_call_count": count_relay_calls(relay_mod["main"].body),
        "relay_text_num_lines": int(len(relay_text.splitlines())),
        "tvm_graph_num_nodes": int(len(graph_json.get("nodes", []))),
        "tvm_graph_num_arg_nodes": int(len(graph_json.get("arg_nodes", []))),
        "tvm_graph_num_heads": int(len(graph_json.get("heads", []))),
        "tvm_artifact_sizes": tvm_artifact_sizes,
    }

In [ ]:
total_samples = 0
onnx_correct = 0
tvm_correct = 0
matching_predictions = 0
max_abs_diff = 0.0
sum_abs_diff = 0.0
num_logits_compared = 0
mismatch_examples = []

for batch_index, (inputs, targets) in enumerate(test_loader):
    inputs_np = inputs.numpy().astype(np.float32)
    targets_np = targets.numpy()

    onnx_logits = run_onnx(inputs_np)
    tvm_logits = run_tvm(inputs_np)

    onnx_preds = np.argmax(onnx_logits, axis=1)
    tvm_preds = np.argmax(tvm_logits, axis=1)

    onnx_correct += int((onnx_preds == targets_np).sum())
    tvm_correct += int((tvm_preds == targets_np).sum())
    matching_predictions += int((onnx_preds == tvm_preds).sum())
    total_samples += int(targets_np.shape[0])

    abs_diff = np.abs(onnx_logits - tvm_logits)
    batch_max_diff = float(np.max(abs_diff))
    max_abs_diff = max(max_abs_diff, batch_max_diff)
    sum_abs_diff += float(np.sum(abs_diff))
    num_logits_compared += int(abs_diff.size)

    if len(mismatch_examples) < 10:
        mismatch_indices = np.where(onnx_preds != tvm_preds)[0]
        for idx in mismatch_indices:
            mismatch_examples.append({
                "batch_index": int(batch_index),
                "sample_in_batch": int(idx),
                "target": int(targets_np[idx]),
                "onnx_pred": int(onnx_preds[idx]),
                "tvm_pred": int(tvm_preds[idx]),
            })
            if len(mismatch_examples) >= 10:
                break

mean_abs_diff = sum_abs_diff / num_logits_compared

results = {
    "comparison": "onnx_runtime_vs_tvm",
    "onnx_model_path": str(ONNX_PATH.relative_to(PROJECT_ROOT)),
    "target": TARGET,
    "opt_level": OPT_LEVEL,
    "batch_size_eval": BATCH_SIZE_EVAL,
    "model_summary": build_model_summary(),
    "num_samples": total_samples,
    "onnx_accuracy": onnx_correct / total_samples,
    "tvm_accuracy": tvm_correct / total_samples,
    "prediction_match_rate": matching_predictions / total_samples,
    "max_abs_logit_difference": max_abs_diff,
    "mean_abs_logit_difference": mean_abs_diff,
    "num_prediction_mismatches": total_samples - matching_predictions,
    "mismatch_examples": mismatch_examples,
    "tvm_artifacts": {
        "library_path": str(TVM_LIB_PATH.relative_to(PROJECT_ROOT)),
        "graph_path": str(TVM_GRAPH_PATH.relative_to(PROJECT_ROOT)),
        "params_path": str(TVM_PARAMS_PATH.relative_to(PROJECT_ROOT)),
        "relay_text_path": str(TVM_RELAY_PATH.relative_to(PROJECT_ROOT)),
    },
}

results

In [ ]:
def summarize_timings(durations: list[float], batch_size: int) -> dict:
    durations_np = np.array(durations, dtype=np.float64)
    total_duration = float(np.sum(durations_np))
    total_samples = int(len(durations) * batch_size)
    return {
        "num_runs": int(len(durations)),
        "num_samples": total_samples,
        "mean_latency_ms": float(np.mean(durations_np) * 1000.0),
        "median_latency_ms": float(np.median(durations_np) * 1000.0),
        "p95_latency_ms": float(np.percentile(durations_np, 95) * 1000.0),
        "throughput_samples_per_sec": float(total_samples / total_duration),
    }

def time_backend(run_fn, timing_inputs: list[np.ndarray], warmup_runs: int) -> list[float]:
    for inputs_np in timing_inputs[:warmup_runs]:
        run_fn(inputs_np)

    durations = []
    for inputs_np in timing_inputs:
        start = time.perf_counter()
        run_fn(inputs_np)
        durations.append(time.perf_counter() - start)
    return durations

timing_inputs = []
for inputs, _ in test_loader:
    timing_inputs.append(inputs.numpy().astype(np.float32))
    if len(timing_inputs) >= TIMING_NUM_SAMPLES:
        break

onnx_timings = time_backend(run_onnx, timing_inputs, TIMING_NUM_WARMUP_RUNS)
tvm_timings = time_backend(run_tvm, timing_inputs, TIMING_NUM_WARMUP_RUNS)

results["performance"] = {
    "timing_num_samples": len(timing_inputs),
    "timing_num_warmup_runs": TIMING_NUM_WARMUP_RUNS,
    "onnx_runtime": summarize_timings(onnx_timings, BATCH_SIZE_EVAL),
    "tvm": summarize_timings(tvm_timings, BATCH_SIZE_EVAL),
}

with RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2)

print(f"Saved ONNX vs TVM verification results to: {RESULTS_PATH}")
print(f"ONNX accuracy: {results['onnx_accuracy']:.4f}")
print(f"TVM accuracy: {results['tvm_accuracy']:.4f}")
print(f"Prediction match rate: {results['prediction_match_rate']:.4f}")
print(f"Max absolute logit difference: {results['max_abs_logit_difference']:.6f}")
print(f"Mean absolute logit difference: {results['mean_abs_logit_difference']:.6f}")
print(f"Prediction mismatches: {results['num_prediction_mismatches']}")
print(f"ONNX mean latency (ms): {results['performance']['onnx_runtime']['mean_latency_ms']:.3f}")
print(f"TVM mean latency (ms): {results['performance']['tvm']['mean_latency_ms']:.3f}")
print(f"ONNX throughput (samples/s): {results['performance']['onnx_runtime']['throughput_samples_per_sec']:.2f}")
print(f"TVM throughput (samples/s): {results['performance']['tvm']['throughput_samples_per_sec']:.2f}")
print(f"ONNX model size (bytes): {results['model_summary']['onnx_file_bytes']}")
print(f"ONNX graph nodes: {results['model_summary']['onnx_num_graph_nodes']}")
print(f"ONNX parameter count: {results['model_summary']['onnx_num_parameters']}")
print(f"TVM compile time (s): {results['model_summary']['tvm_compile_time_s']:.3f}")
print(f"Relay call count: {results['model_summary']['relay_call_count']}")
print(f"Relay text lines: {results['model_summary']['relay_text_num_lines']}")
print(f"TVM graph nodes: {results['model_summary']['tvm_graph_num_nodes']}")
print(f"TVM artifact total size (bytes): {results['model_summary']['tvm_artifact_sizes']['total_bytes']}")
print(f"ONNX op counts: {results['model_summary']['onnx_op_type_counts']}")

## What did the compiler actually change?

Dump pre-opt and post-opt Relay side by side and list the fused kernels in the compiled graph.

In [ ]:
TVM_RELAY_OPT_PATH = TVM_DIR / "lenet_mnist_relay_optimized.txt"

def count_relay_ops(mod) -> Counter:
    op_counts: Counter = Counter()

    class OpCounter(tvm.relay.ExprVisitor):
        def visit_call(self, call):
            if isinstance(call.op, tvm.ir.Op):
                op_counts[call.op.name] += 1
            super().visit_call(call)

    OpCounter().visit(mod["main"])
    return op_counts

with tvm.transform.PassContext(opt_level=OPT_LEVEL):
    optimized_mod, _ = relay.optimize(relay_mod, target=TARGET, params=relay_params)

optimized_relay_text = optimized_mod.astext(show_meta_data=False)
TVM_RELAY_OPT_PATH.write_text(optimized_relay_text, encoding="utf-8")

pre_op_counts = count_relay_ops(relay_mod)
post_op_counts = count_relay_ops(optimized_mod)

graph_json = json.loads(TVM_GRAPH_PATH.read_text(encoding="utf-8"))
fused_kernel_names = [
    node["attrs"]["func_name"]
    for node in graph_json.get("nodes", [])
    if node.get("op") == "tvm_op" and node.get("attrs", {}).get("func_name") != "__nop"
]
unique_fused_kernels = sorted(set(fused_kernel_names))

print(f"Saved optimized Relay to: {TVM_RELAY_OPT_PATH}")
print(f"Pre-opt Relay calls : {sum(pre_op_counts.values())}  ops: {dict(pre_op_counts)}")
print(f"Post-opt Relay calls: {sum(post_op_counts.values())}  ops: {dict(post_op_counts)}")
print(f"Fused kernels in compiled graph: {len(fused_kernel_names)} calls, "
      f"{len(unique_fused_kernels)} unique")
for name in unique_fused_kernels:
    print(f"  - {name}")

In [ ]:
print("=" * 60)
print("PRE-OPTIMIZATION RELAY (as imported from ONNX)")
print("=" * 60)
print(TVM_RELAY_PATH.read_text(encoding="utf-8"))
print()
print("=" * 60)
print("POST-OPTIMIZATION RELAY (after TVM pass pipeline)")
print("=" * 60)
print(optimized_relay_text)